# 06 · Choosing 42 payments to review

The reference cutoff produced 42 validation reviews. I kept those 42 slots fixed and compared two ways to fill them: highest score first, or largest amount among the 84 highest scores. Both queues are selected before I look at the fraud labels.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from google.colab import drive

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/Colab Notebooks/Project 01: Intelligent Financial Fraud Detection")
run_id = "20261121T162317186996Z"
checkpoint = project / "Artifacts" / f"ann_{run_id}.pt"
csv_path = project / "Dataset" / "creditcard.csv"

Mounted at /content/drive


## Rebuilding the validation scores

I use the same middle time window, feature order and saved weights as the dashboard. Original record numbers break score ties consistently.

In [2]:
saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
features = saved["features"]
ann = nn.Sequential(nn.Linear(len(features), 64), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
ann.load_state_dict(saved["state_dict"])
ann.eval()  # Disable dropout while ranking transactions.

data = pd.read_csv(csv_path)
data.index = pd.RangeIndex(1, len(data) + 1, name="source_record")
data = data.sort_values("Time", kind="stable")
start, stop = data.Time.iloc[[int(.70 * len(data)), int(.85 * len(data))]]
window = data.loc[(data.Time >= start) & (data.Time < stop)]
scaled = (window[features].to_numpy() - np.asarray(saved["mean"])) / np.asarray(saved["scale"])
with torch.inference_mode():
    batches = torch.from_numpy(scaled.astype("float32")).split(4096)
    scores = torch.cat([ann(batch).flatten().sigmoid() for batch in batches]).numpy()
candidates = window[["Amount"]].assign(score=scores).reset_index()
k = int((scores >= saved["threshold"]).sum())
assert 0 < k <= len(window)
print(f"{len(window):,} validation records | {k} reviews available")

42,719 validation records | 42 reviews available


In [3]:
# Select both queues using only information available when a transaction is scored.
ranked = candidates.sort_values(["score", "source_record"], ascending=[False, True])
score_queue = ranked.head(k)
shortlist = ranked.head(2 * k)
amount_queue = shortlist.sort_values(["Amount", "score", "source_record"],
                                     ascending=[False, False, True]).head(k)

def record_ids(queue):
    ids = pd.Index(queue.source_record)
    assert ids.is_unique and len(ids) == k
    return ids

baseline = record_ids(score_queue)
challenger = record_ids(amount_queue)
assert set(baseline) == set(candidates.loc[candidates.score >= saved["threshold"], "source_record"])
assert challenger.isin(shortlist.source_record).all()

## What each queue caught

I count confirmed frauds and false alerts, then sum the face value of the frauds sent for review. Each policy has the same workload.

In [4]:
truth = window[["Class", "Amount"]]
all_frauds = truth.loc[truth.Class.eq(1)]

def evaluate(ids):
    reviewed = truth.loc[ids]
    caught = reviewed.loc[reviewed.Class.eq(1)]
    return {"Reviews": len(reviewed), "Frauds caught": len(caught),
            "False alerts": len(reviewed) - len(caught),
            "Precision": len(caught) / len(reviewed),
            "Count recall": len(caught) / len(all_frauds),
            "Fraud amount": caught.Amount.sum(),
            "Amount recall": caught.Amount.sum() / all_frauds.Amount.sum()}

comparison = pd.DataFrame([evaluate(baseline), evaluate(challenger)],
                          index=["Score order", "Amount within high risk"])
print(f"Validation frauds: {len(all_frauds)} | Total fraud amount: {all_frauds.Amount.sum():,.2f}")
print(comparison.round(4).to_string())

Validation frauds: 56 | Total fraud amount: 8,397.70
                         Reviews  Frauds caught  False alerts  Precision  Count recall  Fraud amount  Amount recall
Score order                   42             42             0     1.0000        0.7500       7983.85         0.9507
Amount within high risk       42             24            18     0.5714        0.4286       8214.64         0.9782


## Which payments changed places?

The 22 payments that appeared in both queues cannot explain the difference. I compare the 20 dropped payments with the 20 added payments.

In [5]:
groups = {"Kept by both": baseline.intersection(challenger),
          "Dropped by challenger": baseline.difference(challenger),
          "Added by challenger": challenger.difference(baseline)}
rows = []
for group, ids in groups.items():
    records = truth.loc[ids]
    frauds = records.loc[records.Class.eq(1)]
    rows.append({"Group": group, "Reviews": len(records), "Frauds": len(frauds),
                 "Legitimate": len(records) - len(frauds), "Fraud amount": frauds.Amount.sum()})

swaps = pd.DataFrame(rows).set_index("Group")
print(swaps.round(2).to_string())
delta = comparison.iloc[1] - comparison.iloc[0]
np.testing.assert_allclose(delta["Fraud amount"], swaps.iloc[2]["Fraud amount"] - swaps.iloc[1]["Fraud amount"])
print(f"Change at the same budget: {delta['Frauds caught']:+.0f} frauds, "
      f"{delta['False alerts']:+.0f} false alerts, {delta['Fraud amount']:+,.2f} fraud amount.")

                       Reviews  Frauds  Legitimate  Fraud amount
Group                                                           
Kept by both                22      22           0       7968.09
Dropped by challenger       20      20           0         15.76
Added by challenger         20       2          18        246.55
Change at the same budget: -18 frauds, +18 false alerts, +230.79 fraud amount.


## My choice for this workload

I’d keep score order if the team wants to catch as many fraud cases as possible. Amount-first selection gained 230.79 in flagged fraud amount but lost 18 detections at the same workload. That’s a real tradeoff. Establishing which policy saves more money would require investigation costs and recovery data.